# ESM3 Multimodal Protein Language Model

![ESM3 Multimodal Protein Language Model](https://proto-bio.github.io/proto-assets/images/tool/esm3/hero.png)

This notebook demonstrates the three core operations of the ESM3 tool: extracting sequence embeddings, sampling mutations, and scoring sequences. Unlike sequence-only models, ESM3 was trained on multimodal protein data covering sequence, structure, and function, allowing it to capture richer relationships between amino acid identity, three-dimensional geometry, and biological annotation. Even when working with sequence inputs alone, ESM3's representations encode structural and functional context that purely sequence-based models miss.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm3")
display_overview("esm3")
display_docs_section("esm3", "Background")

# ESM3

ESM3 is EvolutionaryScale's generative protein language model, trained jointly over sequence, structure, and function. This toolkit wraps the open `esm3_sm_open_v1` checkpoint to embed, sample masked positions in, and score supplied protein sequences.

In 2025, [Hayes et al.](https://doi.org/10.1126/science.ads0018) introduced ESM3, a generative model from [EvolutionaryScale](https://www.evolutionaryscale.ai) that departs from the encoder-only design of the ESM-1/ESM-2 line. ESM3 is a masked generative transformer that represents a protein across three simultaneous tracks (amino-acid sequence, discrete structure tokens, and function annotation). Training masks spans across all three tracks, so a single model can be prompted with any combination of partial sequence, structure, and function and asked to complete the rest. The flagship 98B-parameter model (`esm3-large-2024-03`) is available through the [EvolutionaryScale Forge](https://forge.evolutionaryscale.ai) API under closed-beta access (also offered via AWS SageMaker); the publicly released open checkpoint, `esm3_sm_open_v1`, is the small 1.4B-parameter variant.

ESM3 is the multimodal successor to ESM-2 ([Lin et al., 2023](https://doi.org/10.1126/science.ade2574)). Where ESM-2 is a sequence-only masked language model, ESM3 adds structure and function tracks and a generative objective. For pure sequence-embedding workloads ESM-2 remains lighter and faster; ESM3 is the choice when masked generative editing matters. This toolkit exposes only the sequence-track operations (embeddings, masked sampling, scoring) over supplied sequences.

## Available tools

In [2]:
display_available_tools("esm3")

- **`run_esm3_embeddings()`** — Extract protein sequence embeddings and logits using ESM3
- **`run_esm3_sample()`** — Sample masked positions in protein sequences using ESM3 language model
- **`run_esm3_score()`** — Score protein sequences using ESM3 language model

### `run_esm3_embeddings`

ESM3 produces dense vector representations for each input protein sequence. Because the model was trained on multimodal data, these embeddings encode not only evolutionary patterns but also structural and functional context. The mean-pooled vectors can be used for similarity search, clustering, classification, and as input features for downstream machine learning models.

In [3]:
from proto_tools import (
    ESM3EmbeddingsConfig,
    ESM3EmbeddingsInput,
    run_esm3_embeddings,
)

In [4]:
# Display docs
display_api_reference("esm3", "input", "run_esm3_embeddings")

# Two hemoglobin-like sequences to embed
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM3EmbeddingsInput(sequences=sequences)

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [5]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_embeddings")

# Configure the open-source ESM3 model | see docs above for all fields
config = ESM3EmbeddingsConfig(
    model_checkpoint="esm3_sm_open_v1",
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM3EmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `model_checkpoint` | `Literal['esm3_sm_open_v1']` | `'esm3_sm_open_v1'` | ESM3 weights variant |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |
| `repr_layer` | `int` | `-1` | Transformer layer index for embeddings; -1 returns post-norm output, others select pre-norm |

In [6]:
# Run the embeddings function
embeddings_result = run_esm3_embeddings(inputs, config)

Running run_esm3_embeddings [00:00]

In [7]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_embeddings")

# Inspect the embedding dimension and first few values for each sequence
for i, emb in enumerate(embeddings_result.results):
    print(f"Sequence {i + 1}: embedding length = {len(emb.mean_embedding)}, first 5 values = {emb.mean_embedding[:5]}")

**Output** — `ESM3EmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.masked_models.shared_data_models.SequenceEmbedding]` | required | Per-sequence embedding results |

Sequence 1: embedding length = 1536, first 5 values = [-0.0255126953125, 0.0400390625, 0.01806640625, -0.07861328125, 0.1201171875]
Sequence 2: embedding length = 1536, first 5 values = [-0.03173828125, 0.027099609375, 0.0177001953125, -0.07958984375, 0.1123046875]


### `run_esm3_sample`

ESM3 sampling uses the model's predicted amino acid distributions to propose mutations at masked positions. Because ESM3 was trained on multimodal data, its position confidence estimates reflect structural and functional plausibility in addition to evolutionary conservation. There are two ways to specify which positions to sample:

1. **Custom masks** — directly mark positions with `_` in the input sequence to control exactly which residues are redesigned
2. **Masking strategy** — let the `MaskingStrategy` config automatically select positions using random, entropy-based, or max-logit methods

#### Approach 1: Custom masks with `_`

Use the `_` character to explicitly mark which positions should be redesigned. All other positions are held fixed. This is useful when you know exactly which residues you want to mutate — for example, redesigning a loop region or a specific binding interface.

In [8]:
from proto_tools import (
    ESM3SampleConfig,
    ESM3SampleInput,
    run_esm3_sample,
)

# Display input docs
display_api_reference("esm3", "input", "run_esm3_sample")

# Mask positions 6-10 with _ to redesign that region
masked_input = ESM3SampleInput(sequences=["MVLSP_____VKAAW"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [9]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_sample")

# No masking strategy needed — positions are already specified by _
config = ESM3SampleConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM3SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `masking_strategy` | `proto_tools.transforms.masking.base.MaskingStrategy` | `MaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None, temperature=1.0)` | Strategy for selecting positions to mask for resampling |
| `model_checkpoint` | `Literal['esm3_sm_open_v1']` | `'esm3_sm_open_v1'` | ESM3 weights variant |
| `sampling_method` | `Literal['single_pass', 'iterative_refinement']` | `'single_pass'` | 'single_pass' samples every mask in one forward; 'iterative_refinement' uses batch_generate |
| `temperature` | `float` | `1.0` | Softmax temperature for per-position amino-acid sampling |
| `top_p` | `float` | `1.0` | Nucleus sampling threshold; 1.0 disables |
| `num_steps` | `int` | `20` | Iterative-refinement decoding steps; diminishing returns above 20 |
| `schedule` | `Literal['cosine', 'linear']` | `'cosine'` | Unmask schedule across rounds; 'cosine' fronts more commits late |
| `strategy` | `Literal['random', 'entropy']` | `'random'` | Position-selection per round; 'entropy' commits the most-confident first |
| `temperature_annealing` | `bool` | `True` | Anneal temperature toward 0 across rounds |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |

In [10]:
# Run sampling on the custom-masked input
custom_mask_result = run_esm3_sample(masked_input, config)

In [11]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_sample")

# Show the original vs sampled — only the masked positions change
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(custom_mask_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

**Output** — `ESM3SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Sampled/restored protein sequences |
| `logits` | `list[list[list[float]]] \| None` | `None` | Per-position amino acid logits. Shape: [num_sequences, seq_len, 20]. |

Original: MVLSPADKTNVKAAW
Sampled:  MVLSPISPSLVKAAW  (mutated positions: [6, 7, 8, 9, 10])


#### Approach 2: Automatic position selection with `MaskingStrategy`

Instead of manually placing `_` characters, you can let `MaskingStrategy` choose which positions to mask. The `method` parameter controls the selection logic:

- **`"random"`** (default) — uniform random selection
- **`"entropy"`** — targets positions where ESM3 is least certain about the original residue
- **`"max-logit"`** — targets positions where ESM3 most confidently predicts a different amino acid

Use `num_mutations` for an exact count or `mask_fraction` for a proportion of positions.

In [12]:
from proto_tools.transforms.masking import MaskingStrategy

# Entropy-based masking to target the 5 most uncertain positions
strategy_input = ESM3SampleInput(sequences=["MVLSPADKTNVKAAW"])
strategy_config = ESM3SampleConfig(
    masking_strategy=MaskingStrategy(method="entropy", num_mutations=5),
    device="cuda",  # Change to "cpu" if no GPU available
)

strategy_result = run_esm3_sample(strategy_input, strategy_config)

In [13]:
# The model automatically selected the 5 highest-entropy positions to mutate
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(strategy_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

Original: MVLSPADKTNVKAAW
Sampled:  MVLSPATKTNPLIAC  (mutated positions: [7, 11, 12, 13, 15])


### `run_esm3_score`

Pseudo-perplexity scoring masks each position in a sequence one at a time and measures how well the model predicts the original residue. Because ESM3 was trained on multimodal data, its perplexity scores reflect not just evolutionary conservation but also structural and functional plausibility. Lower perplexity indicates that the sequence is more consistent with the patterns the model learned during training. This is useful for ranking designed sequences, filtering out poor candidates before experimental validation, or identifying anomalous regions within a protein.

In [14]:
from proto_tools import (
    ESM3ScoringConfig,
    ESM3ScoringInput,
    run_esm3_score,
)

In [15]:
# Display docs
display_api_reference("esm3", "input", "run_esm3_score")

# Two short sequences to score
inputs = ESM3ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

In [16]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_score")

# Process 16 masked variants per forward pass | see docs above for all fields
config = ESM3ScoringConfig(
    batch_size=16,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM3ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int \| None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `batch_size` | `int` | `1` | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| `return_logits` | `bool` | `False` | Include per-position logits in the output (large; disable to save memory) |
| `model_checkpoint` | `Literal['esm3_sm_open_v1']` | `'esm3_sm_open_v1'` | ESM3 weights variant |

In [17]:
# Run the scoring function
score_result = run_esm3_score(inputs, config)

Running run_esm3_score [00:00]

In [18]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_score")

# Show all scoring metrics for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {inputs.sequences[i]}")
    print(f"  Log-likelihood:     {score['log_likelihood']:.3f}")
    print(f"  Avg log-likelihood: {score['avg_log_likelihood']:.3f}")
    print(f"  Perplexity:         {score['perplexity']:.3f}")

**Output** — `MaskedModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `list[proto_tools.tools.masked_models.shared_data_models.MaskedModelScoringMetrics]` | required | List of scoring outputs, one per input sequence |

Sequence 1: MKTAYIAKQR
  Log-likelihood:     -27.789
  Avg log-likelihood: -2.779
  Perplexity:         16.101
Sequence 2: EVQLVESGGS
  Log-likelihood:     -27.750
  Avg log-likelihood: -2.775
  Perplexity:         16.039


## Export Results

Results from each function can be exported to standard file formats for downstream analysis. Embeddings export to CSV with one row per sequence, sampled sequences export to FASTA for compatibility with other bioinformatics tools, and scores export to JSON with full metadata.

In [19]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export("esm3_embeddings", export_path=output_dir, file_format="csv")
print(f"Embeddings exported to {output_dir / 'esm3_embeddings.csv'}")

strategy_result.export("esm3_sequences", export_path=output_dir, file_format="fasta")
print(f"Sampled sequences exported to {output_dir / 'esm3_sequences.fasta'}")

score_result.export("esm3_scores", export_path=output_dir, file_format="json")
print(f"Scores exported to {output_dir / 'esm3_scores.json'}")

Embeddings exported to example_output/esm3_embeddings.csv
Sampled sequences exported to example_output/esm3_sequences.fasta
Scores exported to example_output/esm3_scores.json
